In [ ]:
%run ./_config

In [0]:
# ddl create dimension tables 

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.dim_stores (
        store_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
        source_store_id INT NOT NULL,
        store_name STRING,
        address STRING,
        city STRING,
        zip_code STRING,
        county STRING,
        store_location STRING,
        valid_from DATE NOT NULL,
        valid_to DATE NOT NULL,
        is_current BOOLEAN NOT NULL
    )
    TBLPROPERTIES ('quality' = 'silver', 'scd_type' = '2')
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.dim_products (
        product_id BIGINT GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
        source_product_id INT NOT NULL,
        item_description STRING,
        category_code INT,
        category_name STRING,
        pack INT,
        bottle_volume_ml INT,
        valid_from DATE NOT NULL,
        valid_to DATE NOT NULL,
        is_current BOOLEAN NOT NULL
    )
    TBLPROPERTIES ('quality' = 'silver', 'scd_type' = '2')
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.dim_vendors (
        vendor_id GENERATED ALWAYS AS IDENTITY (START WITH 1 INCREMENT BY 1),
        source_vendor_id INT NOT NULL,
        vendor_name STRING,
        valid_from DATE NOT NULL,
        valid_to DATE NOT NULL,
        is_current BOOLEAN NOT NULL
    )
    TBLPROPERTIES ('quality' = 'silver', 'scd_type' = '2')
""")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.fct_sales (
        invoice_line_no STRING NOT NULL,
        sale_date DATE NOT NULL,
        store_id BIGINT,
        product_id BIGINT,
        vendor_id BIGINT,
        bottles_sold INT,
        bottle_volume_ml INT,
        state_bottle_cost DECIMAL(10,2),
        state_bottle_retail DECIMAL(10,2),
        sale_dollars DECIMAL(12,2),
        volume_sold_liters DECIMAL(10,4),
        profit DECIMAL(12,2)
    )
    TBLPROPERTIES (
        'quality' = 'silver',
        'delta.enableChangeDataFeed' = 'true'
    )
""")

print("DDL complete, silver tables ready")

In [ ]:
# get the most recent attribute combination per business key
# temp views are the source of truth for what the current state should be

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW stg_stores AS
    WITH ranked AS (
        SELECT
            source_store_id,
            store_name,
            address,
            city,
            zip_code,
            county,
            store_location,
            MAX(sale_date) AS last_seen_date,
            ROW_NUMBER() OVER (
                PARTITION BY source_store_id
                ORDER BY MAX(sale_date) DESC
            ) AS rn
        FROM {SILVER_SCHEMA}.cleaned_sales
        GROUP BY source_store_id, store_name, address, city, zip_code, county, store_location
    )
    SELECT source_store_id, store_name, address, city, zip_code, county, store_location, last_seen_date
    FROM ranked
    WHERE rn = 1
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW stg_products AS
    WITH ranked AS (
        SELECT
            source_product_id,
            item_description,
            category_code,
            category_name,
            pack,
            bottle_volume_ml,
            MAX(sale_date) AS last_seen_date,
            ROW_NUMBER() OVER (
                PARTITION BY source_product_id
                ORDER BY MAX(sale_date) DESC
            ) AS rn
        FROM {SILVER_SCHEMA}.cleaned_sales
        GROUP BY source_product_id, item_description, category_code, category_name, pack, bottle_volume_ml
    )
    SELECT source_product_id, item_description, category_code, category_name, pack, bottle_volume_ml, last_seen_date
    FROM ranked
    WHERE rn = 1
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW stg_vendors AS
    WITH ranked AS (
        SELECT
            source_vendor_id,
            vendor_name,
            MAX(sale_date) AS last_seen_date,
            ROW_NUMBER() OVER (
                PARTITION BY source_vendor_id
                ORDER BY MAX(sale_date) DESC
            ) AS rn
        FROM {SILVER_SCHEMA}.cleaned_sales
        GROUP BY source_vendor_id, vendor_name
    )
    SELECT source_vendor_id, vendor_name, last_seen_date
    FROM ranked
    WHERE rn = 1
""")

# verify staging counts
for v in ["stg_stores", "stg_products", "stg_vendors"]:
    cnt = spark.sql(f"SELECT COUNT(*) AS c FROM {v}").collect()[0]["c"]
    print(f"  {v}: {cnt:,} distinct business keys")

In [ ]:
# dim_stores SCD 2 
# step A: expire current records where tracked attributes changed
# step B: insert new current versions 

# STEP A 
expired = spark.sql(f"""
    MERGE INTO {SILVER_SCHEMA}.dim_stores AS tgt
    USING stg_stores AS src
    ON tgt.source_store_id = src.source_store_id AND tgt.is_current = TRUE
    WHEN MATCHED AND (
           tgt.store_name != src.store_name
        OR tgt.address != src.address
        OR tgt.city != src.city
        OR tgt.zip_code != src.zip_code
        OR tgt.county != src.county
    )
    THEN UPDATE SET
        tgt.is_current = FALSE,
        tgt.valid_to = src.last_seen_date
""")

# STEP B 
inserted = spark.sql(f"""
    INSERT INTO {SILVER_SCHEMA}.dim_stores
        (source_store_id, store_name, address, city, zip_code, county, store_location,
         valid_from, valid_to, is_current)
    SELECT
        src.source_store_id,
        src.store_name,
        src.address,
        src.city,
        src.zip_code,
        src.county,
        src.store_location,
        CASE
            WHEN dim.source_store_id IS NULL
                 AND (SELECT COUNT(*) FROM {SILVER_SCHEMA}.dim_stores) = 0
            THEN DATE '2000-01-01'
            ELSE COALESCE(src.last_seen_date, current_date())
        END AS valid_from,
        DATE '9999-12-31' AS valid_to,
        TRUE AS is_current
    FROM stg_stores src
    LEFT JOIN {SILVER_SCHEMA}.dim_stores dim
        ON  src.source_store_id = dim.source_store_id
        AND dim.is_current = TRUE
    WHERE dim.source_store_id IS NULL
""")

print("dim_stores SCD2 complete")

In [ ]:
# dim_products SCD 2

# STEP A
spark.sql(f"""
    MERGE INTO {SILVER_SCHEMA}.dim_products AS tgt
    USING stg_products AS src
    ON tgt.source_product_id = src.source_product_id AND tgt.is_current = TRUE
    WHEN MATCHED AND (
           tgt.item_description != src.item_description
        OR tgt.category_code != src.category_code
        OR tgt.category_name != src.category_name
    )
    THEN UPDATE SET
        tgt.is_current = FALSE,
        tgt.valid_to = src.last_seen_date
""")

# STEP B
spark.sql(f"""
    INSERT INTO {SILVER_SCHEMA}.dim_products
        (source_product_id, item_description, category_code, category_name, pack, bottle_volume_ml,
         valid_from, valid_to, is_current)
    SELECT
        src.source_product_id,
        src.item_description,
        src.category_code,
        src.category_name,
        src.pack,
        src.bottle_volume_ml,
        CASE
            WHEN dim.source_product_id IS NULL
                 AND (SELECT COUNT(*) FROM {SILVER_SCHEMA}.dim_products) = 0
            THEN DATE '2000-01-01'
            ELSE COALESCE(src.last_seen_date, current_date())
        END AS valid_from,
        DATE '9999-12-31'  AS valid_to,
        TRUE AS is_current
    FROM stg_products src
    LEFT JOIN {SILVER_SCHEMA}.dim_products dim
        ON  src.source_product_id = dim.source_product_id
        AND dim.is_current = TRUE
    WHERE dim.source_product_id IS NULL
""")

print("dim_products SCD2 complete")

In [ ]:
# dim_vendors SCD 2

# STEP A
spark.sql(f"""
    MERGE INTO {SILVER_SCHEMA}.dim_vendors AS tgt
    USING stg_vendors AS src
    ON tgt.source_vendor_id = src.source_vendor_id AND tgt.is_current = TRUE
    WHEN MATCHED AND (
        tgt.vendor_name != src.vendor_name
    )
    THEN UPDATE SET
        tgt.is_current = FALSE,
        tgt.valid_to  = src.last_seen_date
""")

# STEP B
spark.sql(f"""
    INSERT INTO {SILVER_SCHEMA}.dim_vendors
        (source_vendor_id, vendor_name, valid_from, valid_to, is_current)
    SELECT
        src.source_vendor_id,
        src.vendor_name,
        CASE
            WHEN dim.source_vendor_id IS NULL
                 AND (SELECT COUNT(*) FROM {SILVER_SCHEMA}.dim_vendors) = 0
            THEN DATE '2000-01-01'
            ELSE COALESCE(src.last_seen_date, current_date())
        END AS valid_from,
        DATE '9999-12-31' AS valid_to,
        TRUE AS is_current
    FROM stg_vendors src
    LEFT JOIN {SILVER_SCHEMA}.dim_vendors dim
        ON  src.source_vendor_id = dim.source_vendor_id
        AND dim.is_current = TRUE
    WHERE dim.source_vendor_id IS NULL
""")

print("dim_vendors SCD2 complete")

In [ ]:
# dimension load summary
for tbl, bk in [("dim_stores", "source_store_id"), ("dim_products", "source_product_id"), ("dim_vendors", "source_vendor_id")]:
    stats = spark.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN is_current THEN 1 ELSE 0 END) AS current_rows,
            SUM(CASE WHEN NOT is_current THEN 1 ELSE 0 END) AS expired_rows,
            COUNT(DISTINCT {bk}) AS distinct_bks
        FROM {SILVER_SCHEMA}.{tbl}
    """).collect()[0]
    print(f"{tbl}: {stats['total_rows']:,} total | {stats['current_rows']:,} current | "
          f"{stats['expired_rows']:,} expired | {stats['distinct_bks']:,} business keys")

In [ ]:
# fct_sales 
# joins cleaned_sales to current dim records
# merge on invoice_line_no handles deduplication, insert only if new record

result = spark.sql(f"""
    MERGE INTO {SILVER_SCHEMA}.fct_sales AS tgt
    USING (
        SELECT
            c.invoice_line_no,
            c.sale_date,
            s.store_id,
            p.product_id,
            v.vendor_id,
            c.bottles_sold,
            c.bottle_volume_ml,
            c.state_bottle_cost,
            c.state_bottle_retail,
            c.sale_dollars,
            c.volume_sold_liters,
            c.sale_dollars - (c.state_bottle_cost * c.bottles_sold) AS profit
        FROM {SILVER_SCHEMA}.cleaned_sales c
        LEFT JOIN {SILVER_SCHEMA}.dim_stores s
            ON  c.source_store_id = s.source_store_id
            AND c.sale_date >= s.valid_from
            AND c.sale_date < s.valid_to
        LEFT JOIN {SILVER_SCHEMA}.dim_products p
            ON  c.source_product_id = p.source_product_id
            AND c.sale_date >= p.valid_from
            AND c.sale_date < p.valid_to
        LEFT JOIN {SILVER_SCHEMA}.dim_vendors v
            ON  c.source_vendor_id = v.source_vendor_id
            AND c.sale_date >= v.valid_from
            AND c.sale_date < v.valid_to
    ) AS src
    ON tgt.invoice_line_no = src.invoice_line_no
    WHEN NOT MATCHED THEN INSERT *
""")

print("fct_sales load complete")

In [ ]:
# fact load summary
fact_stats = spark.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN store_id   IS NULL THEN 1 ELSE 0 END) AS null_store_fk,
        SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_fk,
        SUM(CASE WHEN vendor_id  IS NULL THEN 1 ELSE 0 END) AS null_vendor_fk,
        MIN(sale_date) AS earliest_sale,
        MAX(sale_date) AS latest_sale
    FROM {SILVER_SCHEMA}.fct_sales
""").collect()[0]

print(f"fct_sales: {fact_stats['total_rows']:,} rows")
print(f"  Date range: {fact_stats['earliest_sale']} to {fact_stats['latest_sale']}")
print(f"  NULL store_id:   {fact_stats['null_store_fk']:,}")
print(f"  NULL product_id: {fact_stats['null_product_fk']:,}")
print(f"  NULL vendor_id:  {fact_stats['null_vendor_fk']:,}")

if fact_stats['null_store_fk'] > 0 or fact_stats['null_product_fk'] > 0 or fact_stats['null_vendor_fk'] > 0:
    print("\nNULL surrogate keys detected")

In [ ]:
# pass load stats to downstream tasks
try:
    dbutils.jobs.taskValues.set(key="fact_rows", value=int(fact_stats['total_rows']))
    dbutils.jobs.taskValues.set(key="null_fks",
        value=int(fact_stats['null_store_fk'] + fact_stats['null_product_fk'] + fact_stats['null_vendor_fk']))
except Exception:
    pass  # not running in a Workflow

print("Silver layer load complete")